_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._

# External Merge Sort

External merge sort sorts a table R that does not fit into main memory: **pass 0** writes sorted **runs** to disk,
**merges** combine them. This notebook runs ExplainDB's `ExternalMergeSort`
([`sorting.py`](../system/query_processing/sorting.py)) and measures what it does (lecture 09b, slides 22 and 23).

- Tutorial page: [External Sorting & Queues](<../docs/tutorial/03 Query Processing/External Sorting & Queues.md>)
- Film of the same sort, seen from above:
  [external-sorting](https://htmlpreview.github.io/?https://github.com/explaindb/explaindb/blob/main/visualizations/external-sorting.html)

In [1]:
import sys, pathlib

# Put the repo root (the directory containing `system/`) on sys.path so
# `from system...` works whether this notebook is opened from its own
# folder or from the repo root.
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "system").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

## Terms and symbols

N and M count **elements**, not pages, as in Vitter, *Algorithms and Data Structures for External Memory* (2008).

| Term or symbol | Meaning |
| --- | --- |
| element | one value to sort; here an integer that stands for the key of a tuple |
| R | the input table, as on the slides |
| N | the number of elements of R |
| M | the number of elements that fit in main memory |
| run | a sorted sequence of elements on disk |
| pass 0 | reads R once and writes it as runs, with quicksort or replacement selection |
| quicksort | the slides' way of making runs: sort M elements at a time in memory, giving runs of M, the last maybe shorter. The code sorts with Python's `list.sort()` instead (`RunGenerator.create_runs`) |
| replacement selection | a way of making runs in pass 0; its runs can be longer than M: about 2M on random input, one run on ascending input; see the [Replacement Selection Playground](../notebooks/Replacement-Selection-Playground.ipynb) |
| W | the number of runs pass 0 makes, as in Graefe, *Query Evaluation Techniques for Large Databases* (1993) |
| merge | reads several runs and writes them as one longer run |
| fan-in, F | the most runs one merge reads |
| level | 0 for a run of pass 0; a merge is one level above the highest run it reads (PASS k on the slides) |
| final merge | the last merge: it reads all runs left (at most F) and writes nothing, as its output goes straight to the reader of the sort. *Merges before final* counts the other merges |
| written + read | elements written plus elements read, in pass 0 and all merges: **measured** |
| formula | 2N(1 + ⌈log_F W⌉): 2N for pass 0 and for each of ⌈log_F W⌉ merge passes, where a pass merges all runs in groups of F (**computed**) |

`sort_counted` and `compare` ([`sort_counting.py`](../system/playground/sort_counting.py)) run the **real**
`ExternalMergeSort` on queues that count every element written and read and keep runs in memory (only section 3
sorts on disk). Only the formula's value is computed.

## Data

Every input is a sequence of integers.

| Variable | N | How it is made | Used in section (M; fan-in) |
| --- | --- | --- | --- |
| `random_values` | 10,000 | random integers from 0 to 999,999, fixed seed | 1; 3 and 4 (1,000; 4); 5 (100, 500 and 1,000; 2, 4 and 10) |
| `input_orders` | 3 × 10,000 | `random_values` as it is (random), ascending, descending | 6 (500; 10) |
| `SMALL_EXAMPLE` | 36 | one-digit values, chosen by hand after the slides' example, for fan-in 3 | 2 (4; 2 and 3); 4 (4; 3) |
| `SLIDE_VALUES` | 16 | the values of slides 22 and 23 | 2 (commented out); 4 (4 and 16; 2) |
| `SLIDE_VALUES_SWAPPED` | 16 | `SLIDE_VALUES` with the 1 and the 7 of its last four values swapped | 6 (4; 2) |
| `ASCENDING_16`, `ASCENDING_20`, `ASCENDING_104` | 16, 20, 104 | 0, 1, 2, ... | 4 (4; 2 and 3) |

In [2]:
import random
from collections.abc import Sequence

import pandas as pd

# 10,000 random integers from 0 to 999,999; the fixed seed gives the same values in every run:
rng = random.Random(42)
random_values: list[int] = [rng.randrange(1_000_000) for _ in range(10_000)]
# the same 10,000 values in three orders:
input_orders: dict[str, list[int]] = {
    "random": random_values,
    "ascending": sorted(random_values),
    "descending": sorted(random_values, reverse=True),
}
# 36 one-digit values, written in two rows of 18:
SMALL_EXAMPLE: tuple[int, ...] = (
    4, 5, 1, 4, 2, 5, 7, 5, 6, 5, 5, 2, 2, 1, 9, 8, 8, 6,
    2, 4, 5, 1, 6, 7, 4, 9, 8, 2, 4, 1, 3, 9, 4, 6, 6, 5,
)  # fmt: skip
# the 16 values of slides 22 and 23, and the same values with the 1 and the 7 near the end swapped:
SLIDE_VALUES: tuple[int, ...] = (3, 4, 6, 2, 9, 4, 8, 7, 5, 6, 3, 1, 1, 4, 6, 7)
SLIDE_VALUES_SWAPPED: tuple[int, ...] = (3, 4, 6, 2, 9, 4, 8, 7, 5, 6, 3, 1, 7, 4, 6, 1)
# 0, 1, 2, ... in ascending order:
ASCENDING_16: tuple[int, ...] = tuple(range(16))
ASCENDING_20: tuple[int, ...] = tuple(range(20))
ASCENDING_104: tuple[int, ...] = tuple(range(104))

# every data set by the name of its variable:
DATA_SETS: dict[str, Sequence[int]] = {
    "random_values": random_values,
    "SMALL_EXAMPLE": SMALL_EXAMPLE,
    "SLIDE_VALUES": SLIDE_VALUES,
    "SLIDE_VALUES_SWAPPED": SLIDE_VALUES_SWAPPED,
    "ASCENDING_16": ASCENDING_16,
    "ASCENDING_20": ASCENDING_20,
    "ASCENDING_104": ASCENDING_104,
}
pd.DataFrame(
    [
        {
            "variable": name,
            "N": len(values),
            "first values": ", ".join(map(str, values[:8])) + ", ...",
        }
        for name, values in DATA_SETS.items()
    ]
)

,variable,N,first values
0,random_values,10000,"670487, 116739, 26225, 777572, 288389, 256787,..."
1,SMALL_EXAMPLE,36,"4, 5, 1, 4, 2, 5, 7, 5, ..."
2,SLIDE_VALUES,16,"3, 4, 6, 2, 9, 4, 8, 7, ..."
3,SLIDE_VALUES_SWAPPED,16,"3, 4, 6, 2, 9, 4, 8, 7, ..."
4,ASCENDING_16,16,"0, 1, 2, 3, 4, 5, 6, 7, ..."
5,ASCENDING_20,20,"0, 1, 2, 3, 4, 5, 6, 7, ..."
6,ASCENDING_104,104,"0, 1, 2, 3, 4, 5, 6, 7, ..."


In [3]:
from system.playground.merge_tree import merge_tree_svg, page_labels
from system.playground.sort_counting import (
    Comparison,
    SortCounts,
    compare,
    sort_counted,
)

# the two ways of making runs in pass 0, and the header of the column with the formula's value:
QUICKSORT: str = "quicksort"
REPLACEMENT_SELECTION: str = "replacement selection"
FORMULA: str = "formula 2N(1 + ⌈log_F W⌉)"


def both_ways(sorts: Comparison[int]) -> tuple[tuple[str, SortCounts[int]], ...]:
    """The two sorts of a comparison, each with the name of its way of pass 0.

    @param sorts: the result of `compare`
    @return: (name, counts) for quicksort (`sorts.chunks`: the code sorts chunks of M elements) and for
    replacement selection
    """
    return (
        (QUICKSORT, sorts.chunks),
        (REPLACEMENT_SELECTION, sorts.replacement_selection),
    )


def summary(name: str, counts: SortCounts[int]) -> dict[str, object]:
    """One table row for a sort: its way of pass 0, W, the merges before the final merge, the elements written and
    read, and the formula's value.

    @param name: the way of pass 0
    @param counts: what the sort did
    @return: the row, by column header
    """
    return {
        "pass 0": name,
        "W": len(counts.runs),
        "merges before final": len(counts.merges),
        "written": counts.written,
        "read": counts.read,
        "written + read": counts.cost,
        FORMULA: counts.textbook_cost,
    }


def pages(values: Sequence[int]) -> str:
    """The values in pages of two, as the slides write them: `2,3 4,6`."""
    return " ".join(page_labels(values))


def show_merges(name: str, counts: SortCounts[int]) -> None:
    """Prints what a sort did: the runs of pass 0 (separated by `|`), every merge before the final merge with its
    level, the runs it read (separated by `+`) and the run it wrote, and the final merge with its level and the
    sorted output.

    @param name: the heading of the printout
    @param counts: what the sort did
    """
    print(f"{name}:")
    print("  PASS 0:", " | ".join(pages(run.elements) for run in counts.runs))
    for merge in counts.merges:
        inputs = " + ".join(pages(run.elements) for run in merge.inputs)
        print(f"  PASS {merge.level}: {inputs}")
        print(f"          -> {pages(merge.output.elements)}")
    final = counts.final_merge
    print(
        f"  PASS {final.level}: final merge of {len(final.inputs)} runs (not written):"
    )
    print(f"          -> {pages(counts.output)}")

## 1 Sorting in main memory

**Goal:** the reference result: every external sort of `random_values` below must give the same list as `sorted()`.

In [4]:
sorted_in_memory: list[int] = sorted(random_values)
print("the five smallest values:", sorted_in_memory[:5])

the five smallest values: [96, 177, 193, 205, 301]


## 2 A small example, merge by merge

**Goal:** follow every run and merge of one sort: `SMALL_EXAMPLE` with M = 4 and fan-in 3 (the slides use fan-in 2).
Values are printed in pages of two, as on the slides.

In [5]:
small_sorts: Comparison[int] = compare(SMALL_EXAMPLE, memory=4, fan_in=3)
for name, counts in both_ways(small_sorts):
    show_merges(name, counts)

quicksort:
  PASS 0: 1,4 4,5 | 2,5 5,7 | 2,5 5,6 | 1,2 8,9 | 2,4 6,8 | 1,5 6,7 | 2,4 8,9 | 1,3 4,9 | 4,5 6,6
  PASS 1: 1,4 4,5 + 2,5 5,7 + 2,5 5,6
          -> 1,2 2,4 4,5 5,5 5,5 6,7
  PASS 1: 1,2 8,9 + 2,4 6,8 + 1,5 6,7
          -> 1,1 2,2 4,5 6,6 7,8 8,9
  PASS 1: 2,4 8,9 + 1,3 4,9 + 4,5 6,6
          -> 1,2 3,4 4,4 5,6 6,8 9,9
  PASS 2: final merge of 3 runs (not written):
          -> 1,1 1,1 2,2 2,2 2,3 4,4 4,4 4,4 5,5 5,5 5,5 5,6 6,6 6,6 7,7 8,8 8,9 9,9
replacement selection:
  PASS 0: 1,2 4,4 5,5 5,5 5,6 7,9 | 1,2 2,2 4,5 6,6 7,8 8,8 9 | 1,1 2,3 4,4 4,5 6,6 9
  PASS 1: final merge of 3 runs (not written):
          -> 1,1 1,1 2,2 2,2 2,3 4,4 4,4 4,4 5,5 5,5 5,5 5,6 6,6 6,6 7,7 8,8 8,9 9,9


The two sorts as merge trees, drawn from the real sort. A column of pages is a run. Arrows lead from the runs a merge
reads to the run it writes; the final merge's dashed output is not written.

In [6]:
from IPython.display import HTML, display

display(
    HTML(
        '<div style="display: flex; flex-wrap: wrap; gap: 32px; align-items: flex-start">'
        + merge_tree_svg(small_sorts.chunks, title="Quicksort, M = 4, fan-in 3")
        + merge_tree_svg(
            small_sorts.replacement_selection,
            title="Replacement selection, M = 4, fan-in 3",
        )
        + "</div>"
    )
)

# The slides' own example (slides 22 and 23): SLIDE_VALUES with M = 4 and fan-in 2. Remove the "# " of the lines
# below to print and draw it:
# slide_sorts = compare(SLIDE_VALUES, memory=4, fan_in=2)
# show_merges("slide 22, quicksort", slide_sorts.chunks)
# show_merges("slide 23, replacement selection", slide_sorts.replacement_selection)
# display(
#     HTML(
#         '<div style="display: flex; flex-wrap: wrap; gap: 32px; align-items: flex-start">'
#         + merge_tree_svg(slide_sorts.chunks, title="Slide 22: quicksort, M = 4, fan-in 2")
#         + merge_tree_svg(slide_sorts.replacement_selection, title="Slide 23: replacement selection, M = 4, fan-in 2")
#         + "</div>"
#     )
# )

In [7]:
pd.DataFrame(
    [
        {
            **summary(name, counts),
            "run lengths": ", ".join(map(str, counts.run_lengths)),
            "level of final merge": counts.final_merge.level,
        }
        for name, counts in both_ways(small_sorts)
    ]
)

,pass 0,W,merges before final,written,read,written + read,formula 2N(1 + ⌈log_F W⌉),run lengths,level of final merge
0,quicksort,9,3,72,108,180,216,"4, 4, 4, 4, 4, 4, 4, 4, 4",2
1,replacement selection,3,0,36,72,108,144,"12, 13, 11",1


- Replacement selection makes 3 runs instead of 9; the final merge reads them at once, which saves the three merges
  at level 1. Its runs are about 3M long only because the values were chosen that way; random input gives about 2M
  (section 6).
- A merge of three runs uses M = 4 as one element per run it reads plus one for the run it writes.

### The effect of the fan-in

The same 9 quicksort runs, merged with fan-in 2 and with fan-in 3:

In [8]:
# the same values and the same runs of pass 0, merged with fan-in 2 and with fan-in 3:
small_sorts_by_fan_in: dict[int, SortCounts[int]] = {
    fan_in: sort_counted(
        SMALL_EXAMPLE, memory=4, fan_in=fan_in, replacement_selection=False
    )
    for fan_in in (2, 3)
}
small_fan_in_table: pd.DataFrame = pd.DataFrame(
    [
        {
            "fan-in": fan_in,
            "W": len(counts.runs),
            "level of final merge": counts.final_merge.level,
            "merges before final": len(counts.merges),
            "written + read": counts.cost,
            FORMULA: counts.textbook_cost,
        }
        for fan_in, counts in small_sorts_by_fan_in.items()
    ]
)
small_fan_in_table

,fan-in,W,level of final merge,merges before final,written + read,formula 2N(1 + ⌈log_F W⌉)
0,2,9,4,7,268,360
1,3,9,2,3,180,216


With fan-in 2, the 9 runs need 7 merges before the final one, not 3, and the final merge is at level 4, not 2:
more is written and read, 268 against 180. Here this level is ⌈log_F W⌉; for W = 26 (section 4) the level is 4,
against ⌈log_3 26⌉ = 3.

## 3 Pass 0 on disk

**Goal:** see the runs as files: `ExternalQueueFactory` makes queues that write to temporary files. Quicksort,
M = `MEMORY` = 1,000, fan-in `FAN_IN` = 4.

In [9]:
import os

from system.query_processing.queues.queue_factories import ExternalQueueFactory
from system.query_processing.sorting import ExternalMergeSort

# M and the fan-in of the large sorts of this section and the next:
MEMORY: int = 1_000
FAN_IN: int = 4
on_disk: ExternalMergeSort[int] = ExternalMergeSort(
    iter(random_values),
    ExternalQueueFactory(),
    number_of_tuples_in_main_memory=MEMORY,
    fan_in=FAN_IN,
)
# pass 0:
on_disk.create_runs()
# `on_disk.heap` holds the sort's runs; each run's queue keeps its temporary file in `queue.file`:
disk_run_sizes: list[int] = sorted(run.size for run in on_disk.heap)
pass_0_files: set[str] = {run.queue.file.name for run in on_disk.heap}
for run in on_disk.heap:
    # write the file object's buffer to the file, so that the size on disk is complete:
    run.queue.file.flush()
print(f"{len(disk_run_sizes)} runs of these sizes:", disk_run_sizes)
print(
    f"{len(pass_0_files)} files of {sum(map(os.path.getsize, pass_0_files)):,} bytes in all"
)

# the merges before the final merge, then the final merge, which reads while its output is consumed:
on_disk.merge()
disk_final_sizes: list[int] = sorted(run.size for run in on_disk.heap)
all_files: set[str] = pass_0_files | {run.queue.file.name for run in on_disk.heap}
on_disk_output: list[int] = list(on_disk)
print("the same list as sorted():", on_disk_output == sorted_in_memory)
print("files left:", sum(os.path.exists(name) for name in all_files))

10 runs of these sizes: [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
10 files of 168,756 bytes in all
the same list as sorted(): True
files left: 0


Pass 0 makes W = ⌈N/M⌉ = 10 runs of 1,000 elements, each in a file of its own, deleted once a merge has read it.

## 4 Written + read against the formula

**Goal:** measure what the sort of `random_values` (M = 1,000, fan-in 4) writes and reads, and compare with the
formula.

In [10]:
large_sorts: Comparison[int] = compare(random_values, memory=MEMORY, fan_in=FAN_IN)
pd.DataFrame([summary(name, counts) for name, counts in both_ways(large_sorts)])

,pass 0,W,merges before final,written,read,written + read,formula 2N(1 + ⌈log_F W⌉)
0,quicksort,10,2,18000,28000,46000,60000
1,replacement selection,6,1,14107,24107,38214,60000


Both sorts write and read less than the formula: 46,000 and 38,214 against 60,000. The formula is the same for
W = 10 and W = 6, as ⌈log_4 10⌉ = ⌈log_4 6⌉ = 2; the sort needs two merges before the final one with quicksort, one
with replacement selection.

### Why they differ

**Goal:** find the reasons in seven small sorts. New columns: **data** (the variable with the input) and
**formula − N** (the formula less the N elements that the final merge does not write).

In [11]:
# each case: (the name of the data set in DATA_SETS, M, fan-in, True for replacement selection):
CASES: tuple[tuple[str, int, int, bool], ...] = (
    ("SMALL_EXAMPLE", 4, 3, False),
    ("SMALL_EXAMPLE", 4, 3, True),
    ("SLIDE_VALUES", 4, 2, False),
    ("ASCENDING_20", 4, 2, False),
    ("ASCENDING_104", 4, 3, False),
    ("ASCENDING_16", 4, 2, True),
    ("SLIDE_VALUES", 16, 2, False),
)
# the sort of each case, by (data, M, pass 0):
cost_cases: dict[tuple[str, int, str], SortCounts[int]] = {
    (data, memory, REPLACEMENT_SELECTION if selection else QUICKSORT): sort_counted(
        DATA_SETS[data], memory, fan_in, selection
    )
    for data, memory, fan_in, selection in CASES
}
pd.DataFrame(
    [
        {
            "data": data,
            "pass 0": pass_0,
            "N": len(counts.values),
            "M": counts.memory,
            "fan-in": counts.fan_in,
            "W": len(counts.runs),
            "written + read": counts.cost,
            "formula − N": counts.textbook_cost - len(counts.values),
            FORMULA: counts.textbook_cost,
        }
        for (data, _, pass_0), counts in cost_cases.items()
    ]
)

,data,pass 0,N,M,fan-in,W,written + read,formula − N,formula 2N(1 + ⌈log_F W⌉)
0,SMALL_EXAMPLE,quicksort,36,4,3,9,180,180,216
1,SMALL_EXAMPLE,replacement selection,36,4,3,3,108,108,144
2,SLIDE_VALUES,quicksort,16,4,2,4,80,80,96
3,ASCENDING_20,quicksort,20,4,2,5,116,140,160
4,ASCENDING_104,quicksort,104,4,3,26,752,728,832
5,ASCENDING_16,replacement selection,16,4,2,1,48,16,32
6,SLIDE_VALUES,quicksort,16,16,2,1,48,16,32


The sort differs from the formula in three ways:

| # | The sort ... | Effect on written + read | Rows |
| --- | --- | --- | --- |
| 1 | writes nothing in the final merge | formula − N; the only difference if the runs are equally long and W = F^k (k ≥ 1) | 0 and 2 (W = 3² and W = 2²), e.g. row 0: 180 = 216 − 36; also row 1, whose 3 runs the final merge reads at once |
| 2 | merges the smallest runs first, F at a time, not in fixed passes; the last merge before the final one reads just enough runs to leave F | if W is not a power of F, an element can go through fewer or more merges than ⌈log_F W⌉ | fewer, row 3 (W = 5): runs skip a level, 116 against 140; more, row 4 (W = 26): 12 elements go through four merges instead of three, 752 against 728 |
| 3 | reads a single run (W = 1) once more, in the final merge | 3N instead of 2N | 5 and 6: 48 against 32 |

## 5 Less memory, larger fan-in

**Goal:** change M and the fan-in one at a time, on `random_values` with quicksort. First M = 100, 500 and 1,000 at
fan-in 10:

In [12]:
# M varies, the fan-in stays 10. For each M, the values are sorted with both ways of pass 0; the table shows
# quicksort:
sorts_by_memory: dict[int, Comparison[int]] = {
    memory: compare(random_values, memory, fan_in=10) for memory in (100, 500, 1_000)
}
by_memory: pd.DataFrame = pd.DataFrame(
    [
        {"M": memory, "fan-in": sorts.fan_in, **summary(QUICKSORT, sorts.chunks)}
        for memory, sorts in sorts_by_memory.items()
    ]
)
by_memory

,M,fan-in,pass 0,W,merges before final,written,read,written + read,formula 2N(1 + ⌈log_F W⌉)
0,100,10,quicksort,100,10,20000,30000,50000,60000
1,500,10,quicksort,20,2,16000,26000,42000,60000
2,1000,10,quicksort,10,0,10000,20000,30000,40000


Less memory: more runs and more merges. With M = 1,000, W = 10 is at most the fan-in: only the final merge is left,
with 3N = 30,000 written and read.

Then fan-in 2, 4 and 10 at M = 100:

In [13]:
# the fan-in varies, M stays 100. For each fan-in, the values are sorted with both ways of pass 0; the table shows
# quicksort:
sorts_by_fan_in: dict[int, Comparison[int]] = {
    fan_in: compare(random_values, memory=100, fan_in=fan_in) for fan_in in (2, 4, 10)
}
by_fan_in: pd.DataFrame = pd.DataFrame(
    [
        {"M": sorts.memory, "fan-in": fan_in, **summary(QUICKSORT, sorts.chunks)}
        for fan_in, sorts in sorts_by_fan_in.items()
    ]
)
by_fan_in

,M,fan-in,pass 0,W,merges before final,written,read,written + read,formula 2N(1 + ⌈log_F W⌉)
0,100,2,quicksort,100,98,67200,77200,144400,160000
1,100,4,quicksort,100,32,34800,44800,79600,100000
2,100,10,quicksort,100,10,20000,30000,50000,60000


A larger fan-in: fewer merges for the same 100 runs. In both tables, written + read is below the formula; this is
not a rule (see the single run in section 4).

Replacement selection (`.replacement_selection` of `sorts_by_memory` and `sorts_by_fan_in`) makes 51 runs at
M = 100: 16 merges before the final one at fan-in 4, 5 at fan-in 10. At M = 1,000 and fan-in 10 it saves nothing:
both have only the final merge.

## 6 Replacement selection and the order of the input

**Goal:** see when replacement selection saves merges: `input_orders` with M = 500 and fan-in 10.

In [14]:
sorts_by_order: dict[str, Comparison[int]] = {
    order: compare(values, memory=500, fan_in=10)
    for order, values in input_orders.items()
}
pd.DataFrame(
    [
        {
            "input": order,
            **summary(name, counts),
            "longest run": max(counts.run_lengths),
        }
        for order, sorts in sorts_by_order.items()
        for name, counts in both_ways(sorts)
    ]
)

,input,pass 0,W,merges before final,written,read,written + read,formula 2N(1 + ⌈log_F W⌉),longest run
0,random,quicksort,20,2,16000,26000,42000,60000,500
1,random,replacement selection,11,1,11096,21096,32192,60000,1019
2,ascending,quicksort,20,2,16000,26000,42000,60000,500
3,ascending,replacement selection,1,0,10000,20000,30000,20000,10000
4,descending,quicksort,20,2,16000,26000,42000,60000,500
5,descending,replacement selection,20,2,16000,26000,42000,60000,500


| Input | Runs of replacement selection | Effect |
| --- | --- | --- |
| random | about 2M = 1,000 long: 11 runs instead of 20 | one merge before the final one instead of two |
| ascending | one run of all 10,000 elements | no merge before the final one, which reads the run again: 3N = 30,000 against the formula's 20,000 |
| descending | exactly M = 500 long: an element smaller than the one just written waits for the next run | the same runs and merges as quicksort |

### Longer runs do not always save a level

**Goal:** a counterexample. `SLIDE_VALUES_SWAPPED` with M = 4 and fan-in 2:

In [15]:
swapped_sorts: Comparison[int] = compare(SLIDE_VALUES_SWAPPED, memory=4, fan_in=2)
pd.DataFrame(
    [
        {
            "pass 0": name,
            "run lengths": ", ".join(map(str, counts.run_lengths)),
            "merges before final": len(counts.merges),
            "level of final merge": counts.final_merge.level,
        }
        for name, counts in both_ways(swapped_sorts)
    ]
)

,pass 0,run lengths,merges before final,level of final merge
0,quicksort,"4, 4, 4, 4",2,2
1,replacement selection,"8, 7, 1",1,2


Three runs (8, 7 and 1) instead of four runs of 4 save a merge (1 against 2), but no level: both final
merges are at level 2.

## Tests

The tests check that every external sort returns its input in order, and every number quoted above.

In [16]:
import math
import re
import statistics
import unittest

from system.playground.merge_tree import MAX_VALUES
from system.playground.notebook_tests import run_notebook_tests
from system.playground.sort_counting import CountingQueueFactory, merge_passes
from system.query_processing.queues.queue_factories import ListQueueFactory


class ExternalMergeSortTest(unittest.TestCase):
    """The sorts of this notebook, the numbers its text quotes and the settings of the code it relies on."""

    # the header of the column with the number of merges before the final merge:
    MERGES: str = "merges before final"

    def test_data_sets(self) -> None:
        """The sizes and the making of the data sets, as the table in *Data* gives them."""
        # N of every data set:
        self.assertEqual(
            {
                "random_values": 10_000,
                "SMALL_EXAMPLE": 36,
                "SLIDE_VALUES": 16,
                "SLIDE_VALUES_SWAPPED": 16,
                "ASCENDING_16": 16,
                "ASCENDING_20": 20,
                "ASCENDING_104": 104,
            },
            {name: len(values) for name, values in DATA_SETS.items()},
        )
        # random integers from 0 to 999,999; one-digit values in the small example:
        self.assertTrue(all(0 <= value <= 999_999 for value in random_values))
        self.assertTrue(all(0 <= value <= 9 for value in SMALL_EXAMPLE))
        # the three input orders hold the same 10,000 values: as they are, ascending and descending:
        self.assertEqual(["random", "ascending", "descending"], list(input_orders))
        self.assertIs(random_values, input_orders["random"])
        self.assertEqual(sorted(random_values), input_orders["ascending"])
        self.assertEqual(input_orders["ascending"], input_orders["descending"][::-1])
        # the slides' values, as the slides write them, and the same with the 1 and the 7 of the last four values
        # swapped:
        self.assertEqual("3,4 6,2 9,4 8,7 5,6 3,1 1,4 6,7", pages(SLIDE_VALUES))
        self.assertEqual("3,4 6,2 9,4 8,7 5,6 3,1 7,4 6,1", pages(SLIDE_VALUES_SWAPPED))
        self.assertEqual(SLIDE_VALUES[:12], SLIDE_VALUES_SWAPPED[:12])
        # the ascending inputs count up from 0:
        for name in ("ASCENDING_16", "ASCENDING_20", "ASCENDING_104"):
            self.assertEqual(tuple(range(len(DATA_SETS[name]))), DATA_SETS[name], name)
        # the cost table sorts five of the six small data sets in seven cases, with M = 4 or 16 and fan-in 2 or 3:
        self.assertEqual(7, len(cost_cases))
        self.assertEqual(5, len({data for data, _, _, _ in CASES}))
        self.assertNotIn("random_values", {data for data, _, _, _ in CASES})
        self.assertEqual({4, 16}, {memory for _, memory, _, _ in CASES})
        self.assertEqual({2, 3}, {fan_in for _, _, fan_in, _ in CASES})

    def test_output_is_the_input_in_order(self) -> None:
        """Both ways of pass 0, on list queues and on disk, give back exactly the input values in order, with
        the same run sizes after pass 0 and before the final merge as on counting queues.
        """
        expected = sorted(random_values)
        for factory in (ListQueueFactory, ExternalQueueFactory):
            for selection in (False, True):
                with self.subTest(factory=factory.__name__, selection=selection):
                    counts = (
                        large_sorts.replacement_selection
                        if selection
                        else large_sorts.chunks
                    )
                    # sort the values with M = 1,000 and fan-in 4, with the run sizes after pass 0 of the sort on
                    # counting queues:
                    sort = ExternalMergeSort(
                        iter(random_values),
                        factory(),
                        number_of_tuples_in_main_memory=MEMORY,
                        fan_in=FAN_IN,
                        replacement_selection=selection,
                    )
                    sort.create_runs()
                    self.assertEqual(
                        sorted(counts.run_lengths),
                        sorted(run.size for run in sort.heap),
                    )
                    # the merges, with the run sizes before the final merge of the sort on counting queues:
                    sort.merge()
                    self.assertEqual(
                        sorted(len(run.elements) for run in counts.final_merge.inputs),
                        sorted(run.size for run in sort.heap),
                    )
                    # the same values, in order:
                    self.assertEqual(expected, list(sort))
        # the sort on disk above: the list of sorted(), and no files left:
        self.assertEqual(expected, sorted_in_memory)
        self.assertEqual(sorted_in_memory, on_disk_output)
        self.assertEqual(0, sum(os.path.exists(name) for name in all_files))
        # every sort on counting queues gives its input back in order:
        every_sort = [
            *cost_cases.values(),
            *(counts for _, counts in both_ways(small_sorts)),
            *(counts for _, counts in both_ways(large_sorts)),
            *(counts for _, counts in both_ways(swapped_sorts)),
            *small_sorts_by_fan_in.values(),
            *(
                counts
                for sorts in (
                    *sorts_by_memory.values(),
                    *sorts_by_fan_in.values(),
                    *sorts_by_order.values(),
                )
                for _, counts in both_ways(sorts)
            ),
        ]
        for counts in every_sort:
            self.assertEqual(tuple(sorted(counts.values)), counts.output)

    def test_small_example_with_quicksort(self) -> None:
        """The small example with quicksort: 36 values on 18 pages in 9 runs of 4, three merges of three neighbouring
        runs at level 1 and the final merge of their three outputs at level 2.
        """
        chunks = small_sorts.chunks
        self.assertEqual((4, 3), (small_sorts.memory, small_sorts.fan_in))
        # 36 values on 18 pages; quicksort: 9 runs of 4, three merges of three runs into runs of 12 at level 1:
        self.assertEqual(
            (36, 18), (len(SMALL_EXAMPLE), len(page_labels(SMALL_EXAMPLE)))
        )
        self.assertEqual((4,) * 9, chunks.run_lengths)
        self.assertEqual((12, 12, 12), chunks.merge_outputs)
        self.assertEqual({1}, {merge.level for merge in chunks.merges})
        # three merges of neighbouring runs, and the final merge of their three outputs at level 2:
        self.assertEqual(
            [(0, 1, 2), (3, 4, 5), (6, 7, 8)],
            [tuple(run.number for run in merge.inputs) for merge in chunks.merges],
        )
        self.assertEqual(
            (9, 10, 11), tuple(run.number for run in chunks.final_merge.inputs)
        )
        self.assertEqual(2, chunks.final_merge.level)

    def test_small_example_with_replacement_selection(self) -> None:
        """The small example with replacement selection: 3 runs, on average 3M long, which the final merge reads at
        level 1, with no merge before it.
        """
        selection = small_sorts.replacement_selection
        # replacement selection: 3 runs of 12, 13 and 11, on average 3M = 12 long, read by the final merge at level 1:
        self.assertEqual((12, 13, 11), selection.run_lengths)
        self.assertEqual(3 * 4, statistics.mean(selection.run_lengths))
        # (the final merge reads its runs in the order of the sort's heap of runs, smallest first):
        self.assertEqual(
            [0, 1, 2], sorted(run.number for run in selection.final_merge.inputs)
        )
        self.assertEqual(
            (0, 3, 1),
            (
                len(selection.merges),
                selection.final_fan_in,
                selection.final_merge.level,
            ),
        )

    def test_every_merge_of_the_small_example_reads_three_runs(self) -> None:
        """Every merge in both merge trees of the small example reads three runs, which with fan-in 3 needs an odd
        number of runs, at least 3.
        """
        chunks, selection = small_sorts.chunks, small_sorts.replacement_selection
        # every merge in both trees, the final ones included, reads three runs:
        for counts in (chunks, selection):
            self.assertEqual(
                {3},
                {len(merge.inputs) for merge in (*counts.merges, counts.final_merge)},
            )
        # with fan-in 3, every merge reads three runs only for W = 3, 5, 7 or 9; with an even W (4, 6, 8 or 10),
        # the last merge before the final one reads two:
        for runs in range(1, 11):
            with self.subTest(runs=runs):
                counts = sort_counted(range(4 * runs), 4, 3, False)
                fan_ins = [
                    len(merge.inputs) for merge in (*counts.merges, counts.final_merge)
                ]
                self.assertEqual(runs in (3, 5, 7, 9), set(fan_ins) == {3}, fan_ins)
                if runs in (4, 6, 8, 10):
                    self.assertEqual([2, 3], fan_ins[-2:])

    def test_a_merge_of_three_runs_splits_the_memory_as_3_plus_1(self) -> None:
        """On the real sorter, a merge of three runs of the small example splits M = 4 as 3 + 1."""
        # the memory of a merge of three runs, on the real sorter: each merge gives its three input runs (runs 0-8)
        # one value each and its output run one value when it writes it, 3 + 1 = M = 4; the final merge gives the
        # three outputs (runs 9-11) one value each again:
        factory = CountingQueueFactory()
        sort = ExternalMergeSort(
            iter(SMALL_EXAMPLE),
            factory,
            number_of_tuples_in_main_memory=4,
            fan_in=3,
        )
        sort.create_runs()
        sort.merge()
        self.assertEqual(
            [[1]] * 9 + [[1, 1]] * 3,
            [queue.memory_limits for queue in factory.queues],
        )

    def test_small_example_written_and_read(self) -> None:
        """The elements that the two sorts of the small example write and read, and the sorted output."""
        chunks, selection = small_sorts.chunks, small_sorts.replacement_selection
        # the elements written and read:
        self.assertEqual((72, 108), (chunks.written, chunks.read))
        self.assertEqual((36, 72), (selection.written, selection.read))
        self.assertEqual(tuple(sorted(SMALL_EXAMPLE)), chunks.output)

    def test_limits_of_the_merge_tree_and_the_fan_in(self) -> None:
        """The merge tree is drawn for at most 40 values, and a fan-in outside 2 to M − 1 is rejected with the
        message that names the limits.
        """
        # the merge tree is drawn for at most 40 values, and the fan-in is from 2 to M − 1:
        self.assertEqual(40, MAX_VALUES)
        for fan_in in (1, 4):
            with self.subTest(fan_in=fan_in):
                with self.assertRaisesRegex(
                    ValueError,
                    re.escape(
                        f"the fan-in F must be from 2 to M − 1 = 3, got {fan_in}"
                    ),
                ):
                    compare(SMALL_EXAMPLE, memory=4, fan_in=fan_in)

    def test_fan_in_2_against_fan_in_3(self) -> None:
        """The 9 quicksort runs of the small example merged with fan-in 2 and with fan-in 3, as the table of
        section 2 and the text below it give them.
        """
        # the formula's column header names the number of runs W:
        self.assertEqual("formula 2N(1 + ⌈log_F W⌉)", FORMULA)
        # the table's rows: (fan-in, W, level of final merge, merges before final, written + read, formula):
        self.assertEqual(
            [(2, 9, 4, 7, 268, 360), (3, 9, 2, 3, 180, 216)],
            [tuple(row) for row in small_fan_in_table.itertuples(index=False)],
        )
        self.assertEqual(
            [
                "fan-in",
                "W",
                "level of final merge",
                self.MERGES,
                "written + read",
                FORMULA,
            ],
            list(small_fan_in_table.columns),
        )
        # the rows are measured on two sorts of all 36 values with M = 4 and quicksort, which differ only in the
        # fan-in and so merge the same 9 runs of pass 0:
        self.assertEqual([2, 3], list(small_sorts_by_fan_in))
        for fan_in, counts in small_sorts_by_fan_in.items():
            with self.subTest(fan_in=fan_in):
                self.assertEqual(
                    (SMALL_EXAMPLE, 4, fan_in, False),
                    (
                        counts.values,
                        counts.memory,
                        counts.fan_in,
                        counts.replacement_selection,
                    ),
                )
                self.assertEqual(small_sorts.chunks.runs, counts.runs)
                # the table's row of this fan-in holds the numbers of its sort:
                (row,) = small_fan_in_table[
                    small_fan_in_table["fan-in"] == fan_in
                ].itertuples(index=False)
                self.assertEqual(
                    tuple(row)[1:],
                    (
                        len(counts.runs),
                        counts.final_merge.level,
                        len(counts.merges),
                        counts.cost,
                        counts.textbook_cost,
                    ),
                )
                # the level of the final merge is ⌈log_F W⌉, 4 for fan-in 2 and 2 for fan-in 3:
                self.assertEqual(merge_passes(9, fan_in), counts.final_merge.level)
        # the sort with fan-in 3 is the one that section 2 prints and draws:
        self.assertEqual(small_sorts.chunks, small_sorts_by_fan_in[3])
        # fan-in 2: every merge reads two runs; four merges at level 1, two at level 2, one at level 3, and the
        # final merge at level 4:
        two = small_sorts_by_fan_in[2]
        self.assertEqual(
            {2}, {len(merge.inputs) for merge in (*two.merges, two.final_merge)}
        )
        self.assertEqual([1, 1, 1, 1, 2, 2, 3], [merge.level for merge in two.merges])

    def test_runs_on_disk_and_on_counting_queues(self) -> None:
        """⌈N/M⌉ runs of quicksort, 10 files on disk, and the same sizes on disk and on counting queues."""
        # M = 1,000 and fan-in 4, with a buffer of 100 elements per queue on disk:
        self.assertEqual((1_000, 4), (MEMORY, FAN_IN))
        self.assertEqual(100, ExternalQueueFactory().number_of_objects_in_buffer)
        # pass 0 on disk: ⌈10,000 / 1,000⌉ = 10 runs of 1,000 values, each in a file of its own:
        self.assertEqual([1_000] * 10, disk_run_sizes)
        self.assertEqual(10, len(pass_0_files))
        # the sort on counting queues makes runs of the same sizes and leaves runs of the same sizes for the final
        # merge:
        self.assertEqual(sorted(large_sorts.chunks.run_lengths), disk_run_sizes)
        self.assertEqual(
            sorted(len(run.elements) for run in large_sorts.chunks.final_merge.inputs),
            disk_final_sizes,
        )
        # every number of runs of quicksort in the two tables of section 5 is ⌈N/M⌉:
        for table in (by_memory, by_fan_in):
            self.assertEqual({QUICKSORT}, set(table["pass 0"]))
            for _, row in table.iterrows():
                self.assertEqual(math.ceil(len(random_values) / row["M"]), row["W"])

    def test_written_and_read_against_the_formula(self) -> None:
        """W, the elements written and read and the formula's value of the large sorts, and of every small case
        also its fan-in and formula − N, as the text quotes them.
        """
        # the large sorts (M = 1,000, fan-in 4): W = 10 and 6, two merges before the final one against one, 46,000
        # and 38,214 written and read against the formula's 60,000 for both, as ⌈log_4 10⌉ = ⌈log_4 6⌉ = 2:
        self.assertEqual(
            [(10, 2, 46_000, 60_000), (6, 1, 38_214, 60_000)],
            [
                (
                    len(counts.runs),
                    len(counts.merges),
                    counts.cost,
                    counts.textbook_cost,
                )
                for _, counts in both_ways(large_sorts)
            ],
        )
        self.assertEqual((2, 2), (merge_passes(10, 4), merge_passes(6, 4)))
        # (fan-in, W, written + read, formula − N, formula) of each small case, in the order of the table's rows:
        quoted = {
            ("SMALL_EXAMPLE", 4, QUICKSORT): (3, 9, 180, 180, 216),
            ("SMALL_EXAMPLE", 4, REPLACEMENT_SELECTION): (3, 3, 108, 108, 144),
            ("SLIDE_VALUES", 4, QUICKSORT): (2, 4, 80, 80, 96),
            ("ASCENDING_20", 4, QUICKSORT): (2, 5, 116, 140, 160),
            ("ASCENDING_104", 4, QUICKSORT): (3, 26, 752, 728, 832),
            ("ASCENDING_16", 4, REPLACEMENT_SELECTION): (2, 1, 48, 16, 32),
            ("SLIDE_VALUES", 16, QUICKSORT): (2, 1, 48, 16, 32),
        }
        self.assertEqual(list(quoted), list(cost_cases))
        for case, numbers_quoted in quoted.items():
            with self.subTest(case=case):
                counts = cost_cases[case]
                self.assertEqual(
                    numbers_quoted,
                    (
                        counts.fan_in,
                        len(counts.runs),
                        counts.cost,
                        counts.textbook_cost - len(counts.values),
                        counts.textbook_cost,
                    ),
                )
        # rows 0 and 2: runs of 4 and W a power of the fan-in (9 = 3^2 and 4 = 2^2); row 1: 3 runs, read by the
        # final merge at once:
        self.assertEqual(
            {4}, set(cost_cases[("SMALL_EXAMPLE", 4, QUICKSORT)].run_lengths)
        )
        self.assertEqual(
            {4}, set(cost_cases[("SLIDE_VALUES", 4, QUICKSORT)].run_lengths)
        )
        self.assertEqual(
            (), cost_cases[("SMALL_EXAMPLE", 4, REPLACEMENT_SELECTION)].merges
        )
        # row 3, 20 ascending: a run of pass 0 is merged late, first at level 2, so it skips level 1:
        twenty = cost_cases[("ASCENDING_20", 4, QUICKSORT)]
        self.assertEqual(2, twenty.merges[-1].level)
        self.assertEqual(
            [(4, 0), (8, 1)],
            sorted((len(run.elements), run.level) for run in twenty.merges[-1].inputs),
        )
        # row 4, 104 ascending: four merges for 12 values, against ⌈log_3 26⌉ = 3, so 2 · 12 more than formula − N:
        counts = cost_cases[("ASCENDING_104", 4, QUICKSORT)]
        self.assertEqual((4, 3), (counts.final_merge.level, merge_passes(26, 3)))
        self.assertEqual(2 * 12, counts.cost - (counts.textbook_cost - 104))
        # the 12 values: a run of 12 made at level 1 inside the run of 20 that the last merge before the final one
        # reads, next to a run of 12:
        last = counts.merges[-1]
        self.assertEqual(
            ((12, 1), (20, 2)),
            tuple((len(run.elements), run.level) for run in last.inputs),
        )
        (made_by,) = [
            merge for merge in counts.merges if merge.output == last.inputs[1]
        ]
        self.assertEqual(
            [(4, 0), (4, 0), (12, 1)],
            sorted((len(run.elements), run.level) for run in made_by.inputs),
        )

    def section_5_comparisons(self) -> dict[tuple[int, int], Comparison[int]]:
        """Every sort behind the two tables of section 5.

        @return: the comparisons of `sorts_by_memory` and `sorts_by_fan_in`, by (M, fan-in)
        """
        return {
            **{(memory, 10): sorts for memory, sorts in sorts_by_memory.items()},
            **{(100, fan_in): sorts for fan_in, sorts in sorts_by_fan_in.items()},
        }

    def check_section_5_table(self, table: pd.DataFrame) -> None:
        """Checks that a table of section 5 shows quicksort only, that each row holds the numbers of its sort, and
        that written + read is below the formula in every row.

        @param table: `by_memory` or `by_fan_in`
        """
        comparisons = self.section_5_comparisons()
        # quicksort only, one row per sort:
        self.assertEqual({QUICKSORT}, set(table["pass 0"]))
        # each row of the table holds the numbers of its quicksort sort:
        for _, row in table.iterrows():
            counts = comparisons[(row["M"], row["fan-in"])].chunks
            self.assertEqual(
                (len(counts.runs), len(counts.merges), counts.cost),
                (row["W"], row[self.MERGES], row["written + read"]),
            )
            self.assertEqual(
                (counts.written, counts.read, counts.textbook_cost),
                (row["written"], row["read"], row[FORMULA]),
            )
        # written + read is below the formula, and it equals the sum of the columns written and read:
        self.assertEqual(
            [], table.index[table["written + read"] >= table[FORMULA]].tolist()
        )
        self.assertTrue(
            (table["written"] + table["read"] == table["written + read"]).all()
        )

    def test_sorts_behind_the_tables_of_section_5(self) -> None:
        """The five sorts behind the two tables of section 5 were made with the M and the fan-in of their rows."""
        # every sort behind the tables, by (M, fan-in), with M and the fan-in it was really made with:
        comparisons = self.section_5_comparisons()
        self.assertEqual(5, len(comparisons))
        for (memory, fan_in), sorts in comparisons.items():
            self.assertEqual((memory, fan_in), (sorts.memory, sorts.fan_in))
        # M = 100 at fan-in 10 is in both tables, as the same sort:
        self.assertEqual(sorts_by_memory[100], sorts_by_fan_in[10])

    def test_memory_table_of_section_5(self) -> None:
        """The first table of section 5: M = 100, 500 and 1,000 at fan-in 10, quicksort, and what the text below
        it says about M = 1,000.
        """
        self.check_section_5_table(by_memory)
        self.assertEqual(
            [(100, 10), (500, 10), (1_000, 10)],
            list(zip(by_memory["M"], by_memory["fan-in"])),
        )
        self.assertEqual([100, 500, 1_000], list(sorts_by_memory))
        # M = 1,000, fan-in 10: at most as many runs as the fan-in (10 with quicksort), no merge, 3N written and
        # read for both:
        thousand_10 = self.section_5_comparisons()[(1_000, 10)]
        self.assertEqual(10, len(thousand_10.chunks.runs))
        for _, counts in both_ways(thousand_10):
            self.assertEqual((0, 30_000), (len(counts.merges), counts.cost))
            self.assertLessEqual(len(counts.runs), 10)

    def test_fan_in_table_of_section_5(self) -> None:
        """The second table of section 5: fan-in 2, 4 and 10 at M = 100, quicksort."""
        self.check_section_5_table(by_fan_in)
        self.assertEqual(
            [(100, 2), (100, 4), (100, 10)],
            list(zip(by_fan_in["M"], by_fan_in["fan-in"])),
        )
        self.assertEqual([2, 4, 10], list(sorts_by_fan_in))
        # M = 100, quicksort: 100 runs for every fan-in, and 98, 32 and 10 merges with fan-in 2, 4 and 10:
        self.assertEqual([100, 100, 100], by_fan_in["W"].tolist())
        self.assertEqual([98, 32, 10], by_fan_in[self.MERGES].tolist())

    def test_replacement_selection_in_section_5(self) -> None:
        """The numbers of replacement selection that the text of section 5 quotes from the sorts behind its
        tables.
        """
        comparisons = self.section_5_comparisons()
        # M = 100, quicksort against replacement selection: 100 runs and 32 merges against 51 and 16 (fan-in 4), 10
        # against 5 merges (fan-in 10):
        hundred_4, hundred_10 = comparisons[(100, 4)], comparisons[(100, 10)]
        self.assertEqual(
            (100, 32), (len(hundred_4.chunks.runs), len(hundred_4.chunks.merges))
        )
        self.assertEqual(
            (51, 16),
            (
                len(hundred_4.replacement_selection.runs),
                len(hundred_4.replacement_selection.merges),
            ),
        )
        self.assertEqual(10, len(hundred_10.chunks.merges))
        self.assertEqual(5, len(hundred_10.replacement_selection.merges))
        # replacement selection makes 51 runs at M = 100, whatever the fan-in:
        self.assertEqual(
            {51},
            {
                len(sorts.replacement_selection.runs)
                for sorts in sorts_by_fan_in.values()
            },
        )

    def test_trends_over_memory_and_fan_in(self) -> None:
        """Less memory gives more runs and merges, a larger fan-in fewer merges, and written + read stays below
        the formula, for both ways of pass 0.
        """
        comparisons = self.section_5_comparisons()
        # the less memory, the more runs and merges, for fan-in 10 (the first table) and fan-in 4 and for each way
        # of pass 0; fan-in 4 is not in the first table: M = 500 is sorted here, M = 1,000 in section 4:
        ordered_by_memory: dict[int, list[Comparison[int]]] = {
            10: [comparisons[(memory, 10)] for memory in (100, 500, 1_000)],
            4: [comparisons[(100, 4)], compare(random_values, 500, 4), large_sorts],
        }
        # M = 500 at fan-in 4, quicksort: ⌈N/M⌉ = 20 runs:
        self.assertEqual(20, len(ordered_by_memory[4][1].chunks.runs))
        for fan_in, ordered in ordered_by_memory.items():
            self.assertEqual(
                [(100, fan_in), (500, fan_in), (1_000, fan_in)],
                [(sorts.memory, sorts.fan_in) for sorts in ordered],
            )
            for pass_0 in (QUICKSORT, REPLACEMENT_SELECTION):
                with self.subTest(fan_in=fan_in, pass_0=pass_0):
                    # the three sorts made with this way of pass 0, by rising M:
                    sorts_of_this_way = [
                        dict(both_ways(sorts))[pass_0] for sorts in ordered
                    ]
                    for column, values in (
                        ("W", [len(counts.runs) for counts in sorts_of_this_way]),
                        (
                            self.MERGES,
                            [len(counts.merges) for counts in sorts_of_this_way],
                        ),
                    ):
                        self.assertTrue(
                            all(
                                more > fewer for more, fewer in zip(values, values[1:])
                            ),
                            f"{column} for M = 100, 500, 1,000: {values}",
                        )
        # the larger the fan-in, the fewer merges, for each way of pass 0 (M = 100, fan-in 2, 4 and 10):
        for pass_0 in (QUICKSORT, REPLACEMENT_SELECTION):
            with self.subTest(pass_0=pass_0):
                values = [
                    len(dict(both_ways(sorts))[pass_0].merges)
                    for sorts in sorts_by_fan_in.values()
                ]
                self.assertTrue(
                    all(more > fewer for more, fewer in zip(values, values[1:])),
                    f"{self.MERGES} for fan-in 2, 4, 10: {values}",
                )
        # for both ways of pass 0 of every sort of section 5, and of M = 500 and M = 1,000 at fan-in 4, which are in
        # no table of section 5: written + read is below the formula, and it equals written plus read:
        for case, sorts in (
            *comparisons.items(),
            ((500, 4), ordered_by_memory[4][1]),
            ((1_000, 4), large_sorts),
        ):
            for pass_0, counts in both_ways(sorts):
                with self.subTest(case=case, pass_0=pass_0):
                    self.assertLess(counts.cost, counts.textbook_cost)
                    self.assertEqual(counts.written + counts.read, counts.cost)

    def test_input_orders(self) -> None:
        """Replacement selection on random, ascending and descending input, and on the slides' values with the 1
        and the 7 swapped.
        """
        random_order = sorts_by_order["random"]
        lengths = random_order.replacement_selection.run_lengths
        # M = 500 and fan-in 10 for every order:
        self.assertEqual(
            {(500, 10)},
            {(sorts.memory, sorts.fan_in) for sorts in sorts_by_order.values()},
        )
        # random: runs about 2M = 1,000 long, the first a bit shorter; 11 runs against 20, one merge against two:
        self.assertAlmostEqual(1_000, statistics.mean(lengths[1:-1]), delta=100)
        self.assertLess(lengths[0], 1_000)
        self.assertEqual((11, 20), (len(lengths), len(random_order.chunks.runs)))
        self.assertEqual(
            (1, 2),
            (
                len(random_order.replacement_selection.merges),
                len(random_order.chunks.merges),
            ),
        )
        # ascending: one run of all elements, no merge, 3N against the formula's 2N:
        ascending = sorts_by_order["ascending"].replacement_selection
        self.assertEqual(((10_000,), 0), (ascending.run_lengths, len(ascending.merges)))
        self.assertEqual((30_000, 20_000), (ascending.cost, ascending.textbook_cost))
        # descending: runs of exactly M, as with quicksort, with the same merges and elements written and read:
        descending = sorts_by_order["descending"]
        self.assertEqual({500}, set(descending.replacement_selection.run_lengths))
        for attribute in ("run_lengths", "merge_outputs", "cost"):
            self.assertEqual(
                getattr(descending.chunks, attribute),
                getattr(descending.replacement_selection, attribute),
            )
        # the slides' values with the 1 and the 7 swapped, M = 4 and fan-in 2: runs of 8, 7 and 1 against four
        # runs of 4; one merge before the final one, quicksort: two:
        self.assertEqual((4, 2), (swapped_sorts.memory, swapped_sorts.fan_in))
        self.assertEqual((8, 7, 1), swapped_sorts.replacement_selection.run_lengths)
        self.assertEqual((4, 4, 4, 4), swapped_sorts.chunks.run_lengths)
        self.assertEqual(
            (1, 2),
            (
                len(swapped_sorts.replacement_selection.merges),
                len(swapped_sorts.chunks.merges),
            ),
        )
        # no level saved: both final merges are at level 2, and the formula gives 96 for both:
        for name, counts in both_ways(swapped_sorts):
            with self.subTest(pass_0=name):
                self.assertEqual(2, counts.final_merge.level)
                self.assertEqual(96, counts.textbook_cost)


run_notebook_tests(ExternalMergeSortTest)

test_a_merge_of_three_runs_splits_the_memory_as_3_plus_1 (__main__.ExternalMergeSortTest.test_a_merge_of_three_runs_splits_the_memory_as_3_plus_1)
On the real sorter, a merge of three runs of the small example splits M = 4 as 3 + 1. ... 

ok


test_data_sets (__main__.ExternalMergeSortTest.test_data_sets)
The sizes and the making of the data sets, as the table in *Data* gives them. ... 

ok


test_every_merge_of_the_small_example_reads_three_runs (__main__.ExternalMergeSortTest.test_every_merge_of_the_small_example_reads_three_runs)
Every merge in both merge trees of the small example reads three runs, which with fan-in 3 needs an odd ... 

ok


... (25 outputs left out)

ok


test_written_and_read_against_the_formula (__main__.ExternalMergeSortTest.test_written_and_read_against_the_formula)
W, the elements written and read and the formula's value of the large sorts, and of every small case ... 

ok


----------------------------------------------------------------------
Ran 17 tests in 0.121s

OK


## Things to try

| Try | How | Question |
| --- | --- | --- |
| the slides' own example | remove the `# ` before the code in section 2's drawing cell | which merges differ from the slides? |
| your own values (at most 40) | `mine = compare((9, 8, 7, 6, 5, 4, 3, 2, 1, 0, 9, 8), memory=4, fan_in=3)`, then `show_merges` and `merge_tree_svg` as in section 2 (arrows may cross: smallest runs merge first) | which order gives replacement selection a single run? |
| more memory, larger fan-in | add M = 2,000 to section 5's first table and fan-in 20 to its second (their tests then fail) | what changes at M = 2,000? Why does quicksort at M = 100 always need a merge before the final one? |